# 03 · Explore — the extracted KG is a queryable graph

The knowledge graph isn't hidden behind the retriever — it's a real property graph in AgensGraph. This reuses demo 1's `lightrag_wiki` graph to show the graph-store API and a multi-hop answer.

In [1]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
# LightRAG resets its own logger to INFO on construction, so silence verbose
# INFO/WARNING logs globally (logging.disable can't be overridden by setLevel).
logging.disable(logging.WARNING)
from _common import config
from _common.rag import build_rag
from lightrag import QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status
config.require_openai_key()
rag = build_rag("lightrag_wiki")
await rag.initialize_storages(); await initialize_pipeline_status()
g = rag.chunk_entity_relation_graph
print("total entities:", f"{len(await g.get_all_labels()):,}")

total entities: 15,474


## Most-connected entities (`get_popular_labels` + `node_degree`)

In [2]:
popular = await g.get_popular_labels(limit=12)
for name in popular:
    print(f"  {name:32} degree={await g.node_degree(name)}")

  Armenia                          degree=62
  United States                    degree=51
  Angola                           degree=40
  Animated Television Series       degree=36
  Academy Award for Best Production Design degree=34
  Andhra Pradesh                   degree=33
  Apple II                         degree=33
  Abydos                           degree=31
  All Souls' Day                   degree=31
  Apiaceae                         degree=31
  APL                              degree=31
  Azerbaijan                       degree=31


## Label search (`search_labels`)

In [3]:
print(", ".join(await g.search_labels("Armenia", limit=10)))

Armenia, Armenian, Armenian Alphabet, Armenian Army, Armenian Diaspora, Armenian Drams, Armenian Genocide, Armenian Highlands, Armenian National Movement, Armenian People


## Subgraph export — an ego-network (`get_knowledge_graph`)

In [4]:
hub = popular[0]
kg = await g.get_knowledge_graph(hub, max_depth=2, max_nodes=40)
print(f"around '{hub}': {len(kg.nodes)} nodes, {len(kg.edges)} edges, truncated={kg.is_truncated}")
for e in kg.edges[:10]:
    print(f"  ({e.source}) -[{getattr(e,'type','REL')}]- ({e.target})")

around 'Armenia': 40 nodes, 39 edges, truncated=True
  (Akstafa River) -[DIRECTED]- (Armenia)
  (Armenia) -[DIRECTED]- (Georgia)
  (Armenia) -[DIRECTED]- (Armenian Plateau)
  (Armenia) -[DIRECTED]- (Asian Development Bank)
  (Armenia) -[DIRECTED]- (International Monetary Fund)
  (Armenia) -[DIRECTED]- (Lake Van)
  (Armenia) -[DIRECTED]- (Imports)
  (Armenia) -[DIRECTED]- (Levon Ter-Petrosyan)
  (Armenia) -[DIRECTED]- (Lesser Caucasus)
  (Armenia) -[DIRECTED]- (First Republic of Armenia)


## Multi-hop — connect two hubs (graph vs naive)

In [5]:
e1, e2 = popular[0], popular[1]
q = f"How are '{e1}' and '{e2}' connected? Explain any path between them."
print("Q:", q)
for mode in ["naive", "mix"]:
    ans = await rag.aquery(q, QueryParam(mode=mode, enable_rerank=False))
    print(f"\n### {mode}\n" + str(ans).strip()[:500] + " …")

Q: How are 'Armenia' and 'United States' connected? Explain any path between them.



### naive
The connection between Armenia and the United States can primarily be explored through diplomatic relations, economic ties, and cultural exchanges.

### Diplomatic Relations
Armenia and the United States established formal diplomatic relations following Armenia's independence from the Soviet Union in 1991. The U.S. has been supportive of Armenia in various capacities, promoting democratic reforms and providing aid to assist in the country’s development after the challenges presented by its indep …



### mix
Armenia and the United States have established a relationship marked by diplomacy and international cooperation. The connection between these two nations can be articulated through several key points:

1. **Diplomatic Relations**: Armenia seeks to maintain positive relations with the United States as part of its broader foreign policy that includes fostering ties with both Western and regional powers, including Russia and Iran [1].

2. **Membership in International Organizations**: Armenia is a  …


In [6]:
await rag.finalize_storages()